# Current v3.1 confusion matrices

Two views are available. **Max overlap** gives each GT interval one predicted label: the label with the greatest accumulated overlap. **All overlaps** counts every positively overlapping GT–prediction interval pair; one fragmented prediction can therefore make one GT interval contribute multiple counts. `<no_overlap>` is added once only when a GT interval has no overlapping prediction. These are micro counts, not macro-per-video overlap F1 values. `confusion_summary.csv` also reports pooled class-macro F1 and the constant-majority baseline used in the concise README.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

here = Path.cwd()
archive_dir = here if (here / 'results' / 'current').exists() else here / 'notebooks' / 'archive'
matrix_dir = archive_dir / 'results' / 'current' / 'confusion_matrices'
summary = pd.read_csv(matrix_dir / 'confusion_summary.csv')
pairs = pd.read_csv(matrix_dir / 'confusion_pairs_long.csv')
all_overlap_pairs = pd.read_csv(matrix_dir / 'confusion_all_overlaps_pairs_long.csv')
display(summary.round(3))

In [ ]:
def show_confusion(actor, level, normalized=True, all_overlaps=False):
    suffix = 'row_normalized' if normalized else 'counts'
    view = '_all_overlaps' if all_overlaps else ''
    matrix = pd.read_csv(matrix_dir / f'{actor}_{level}{view}_{suffix}.csv', index_col='gt_label')
    width = min(24, max(7, 0.55 * len(matrix.columns)))
    height = min(30, max(4, 0.42 * len(matrix.index)))
    fig, ax = plt.subplots(figsize=(width, height))
    image = ax.imshow(matrix.values, aspect='auto', cmap='Blues', vmin=0)
    ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=60, ha='right', fontsize=8)
    ax.set_yticks(range(len(matrix.index)), matrix.index, fontsize=8)
    ax.set_xlabel('Predicted overlapping label' if all_overlaps else 'Predicted max-overlap label')
    ax.set_ylabel('GT label')
    title_suffix = suffix.replace('_', ' ')
    view_title = 'all overlaps' if all_overlaps else 'max overlap'
    ax.set_title(f'v3.1 {actor} {level} — {view_title}, {title_suffix}')
    if matrix.size <= 180:
        for row in range(len(matrix.index)):
            for col in range(len(matrix.columns)):
                value = matrix.iat[row, col]
                text = f'{value:.2f}' if normalized else f'{int(value)}'
                ax.text(col, row, text, ha='center', va='center', fontsize=7)
    fig.colorbar(image, ax=ax, shrink=0.8)
    fig.tight_layout()
    return matrix

## Separate actor matrices at the release-score levels

Change the level argument to `raw`, `medium`, or `coarse` to inspect any of the nine complete matrices.

## All-overlap versions

These expose secondary overlapping labels and fragmentation hidden by the max-overlap view. Row percentages describe overlap-pair counts, not a probability per GT interval.

In [ ]:
left_raw_all = show_confusion('left', 'raw', normalized=True, all_overlaps=True)

In [ ]:
right_medium_all = show_confusion('right', 'medium', normalized=True, all_overlaps=True)

In [ ]:
camera_coarse_all = show_confusion('camera', 'coarse', normalized=True, all_overlaps=True)

In [ ]:
left_raw = show_confusion('left', 'raw', normalized=True)

In [ ]:
right_medium = show_confusion('right', 'medium', normalized=True)

In [ ]:
camera_coarse = show_confusion('camera', 'coarse', normalized=True)

In [ ]:
# Largest non-matching cells. Right-raw matches can be off-diagonal because
# its optional contexts use partial unordered overlap rather than equality.
errors = pairs[pairs.evaluator_match_count == 0].sort_values(['actor', 'level', 'count'], ascending=[True, True, False])
display(errors.groupby(['actor', 'level'], group_keys=False).head(10))